# LeetCode 395 — Longest Substring with At Least K Repeating Characters bằng Python

Link đề: https://leetcode.com/problems/longest-substring-with-at-least-k-repeating-characters/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Divide and Conquer, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def longestSubstring(self, s: str, k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **chuỗi con liên tiếp dài nhất** sao cho **mọi ký tự xuất hiện trong nó** đều có tần suất **ít nhất `k`**.
2. Nắm các công cụ Python cần dùng: `collections.Counter` / mảng đếm 26 ô, cắt lát `s[l:r]`, `str.split(c)`, `all(...)`, đệ quy chia để trị, hai con trỏ (cửa sổ trượt).
3. Đi từ cách **vét cạn mọi chuỗi con** (`O(n³)` rồi `O(n²)`) → **chia để trị theo ký tự quá hiếm** (`O(26·n)`) → **cửa sổ trượt với số ký tự khác nhau cố định** (`O(26·n)`).
4. Hiểu hai mấu chốt: (a) một ký tự có tần suất `< k` trong cả đoạn **không thể** nằm trong chuỗi con hợp lệ nào của đoạn đó ⇒ là **điểm cắt**; (b) cửa sổ trượt thông thường **không đơn điệu** với điều kiện "mọi ký tự `≥ k`", nhưng đơn điệu nếu **cố định số ký tự khác nhau tối đa `t`** (thử `t = 1..26`).
5. Tránh các lỗi kinh điển: co cửa sổ theo điều kiện "mọi ký tự `≥ k`" trực tiếp, quên đoạn sau điểm cắt cuối, coi ký tự có tần suất `0` là "hiếm" (đệ quy vô hạn), quên kiểm tra số ký tự đạt ngưỡng.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và số nguyên `k`. Trả về **độ dài** của chuỗi con (liên tiếp) dài nhất của `s` sao cho **tần suất của mỗi ký tự trong chuỗi con đó đều `>= k`**. Nếu không có chuỗi con nào như vậy, trả về `0`.

**Ví dụ 1:** `s = "aaabb"`, `k = 3` → `3`. Chuỗi con dài nhất là `"aaa"`: `a` xuất hiện 3 lần. (`"aaabb"` không được vì `b` chỉ xuất hiện 2 lần.)

**Ví dụ 2:** `s = "ababbc"`, `k = 2` → `5`. Chuỗi con `"ababb"`: `a` xuất hiện 2 lần, `b` xuất hiện 3 lần.

**Ràng buộc:** `1 <= s.length <= 10^4`; `s` chỉ gồm chữ cái thường; `1 <= k <= 10^5`.

Lưu ý: điều kiện áp dụng **cho các ký tự có mặt trong chuỗi con** (ký tự không có mặt không bị ràng buộc).

In [1]:
examples = [
    (("aaabb", 3), 3),
    (("ababbc", 2), 5),
]
for (s, k), expected in examples:
    print(f"s={s!r}, k={k} -> kỳ vọng {expected}")

s='aaabb', k=3 -> kỳ vọng 3
s='ababbc', k=2 -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Đếm tần suất: `Counter` và mảng 26 ô

`collections.Counter(s)` trả về từ điển "ký tự → số lần". Với chữ cái thường ta cũng có thể dùng **mảng 26 ô** và chỉ số `ord(c) - ord('a')` (`ord` trả mã số của ký tự: `ord('a') = 97`).

In [2]:
from collections import Counter

cnt = Counter("ababbc")
print(cnt)                                # Counter({'b': 3, 'a': 2, 'c': 1})
print(cnt['a'], cnt['z'])                 # 2 0   (ký tự vắng mặt cho 0, không lỗi)

arr = [0] * 26
for c in "ababbc":
    arr[ord(c) - ord('a')] += 1
print(arr[:4])                            # [2, 3, 1, 0]

Counter({'b': 3, 'a': 2, 'c': 1})
2 0
[2, 3, 1, 0]


### 3.2. Chuỗi con và cắt lát `s[l:r]`

**Chuỗi con** là đoạn **liên tiếp** của chuỗi. Cắt lát `s[l:r]` lấy các chỉ số `l, l+1, ..., r-1` (**không** gồm `r`); độ dài `r - l`. Với chuỗi dài `n` có `n(n+1)/2` chuỗi con không rỗng.

In [3]:
s = "ababbc"
print(s[1:4], len(s[1:4]))                # bab 3
print(s[0:5])                             # ababb  (chuỗi con dài nhất của ví dụ 2)
n = len(s)
subs = [s[l:r] for l in range(n) for r in range(l + 1, n + 1)]
print(len(subs), n * (n + 1) // 2)        # 21 21

bab 3
ababb
21 21


### 3.3. Điều kiện "mọi ký tự có mặt đều xuất hiện `>= k` lần"

Với một đoạn `t`, tính `Counter(t)` rồi kiểm tra `all(v >= k for v in cnt.values())`. Chỉ xét ký tự **có mặt** (`Counter` không chứa ký tự có tần suất `0`). Với `t` rỗng, `all` trên rỗng là `True` (độ dài `0`, vô hại).

In [4]:
def is_valid(t, k):
    return all(v >= k for v in Counter(t).values())


print(is_valid("ababb", 2), is_valid("aaabb", 3), is_valid("aaa", 3), is_valid("", 5))   # True False True True

True False True True


### 3.4. `str.split(c)`: cắt chuỗi tại một ký tự

`t.split(c)` cắt `t` tại **mọi** ký tự `c` và trả về list các mảnh (bỏ `c`). Hai ký tự `c` liền nhau hoặc `c` ở đầu/cuối sinh ra mảnh **rỗng** (`''`).

In [5]:
print("aabxaab".split("x"))       # ['aab', 'aab']
print("bbaaa".split("b"))         # ['', '', 'aaa']
print("abc".split("z"))           # ['abc']  (không có ký tự cắt)

['aab', 'aab']
['', '', 'aaa']
['abc']


### 3.5. Chia để trị (divide & conquer) trên chuỗi

Cách làm: **chia** bài toán thành các bài nhỏ hơn *độc lập*, giải từng bài, rồi **gộp** (ở đây là lấy `max`). Hàm đệ quy cần **trường hợp cơ sở**: đoạn đủ ngắn để biết ngay đáp án (ví dụ `len(t) < k` thì không thể có ký tự nào xuất hiện `>= k` lần ⇒ đáp án `0`).

In [6]:
def demo_dc(t):
    """Ví dụ nhỏ: độ dài lớn nhất của mảnh khi cắt theo 'x' (chia để trị đơn giản)."""
    if 'x' not in t:
        return len(t)
    return max(demo_dc(piece) for piece in t.split('x'))


print(demo_dc("abxabcxa"))        # 3 (mảnh 'abc')

3


### 3.6. Cửa sổ trượt (hai con trỏ) và tính đơn điệu

**Cửa sổ trượt** giữ đoạn `[l, r]`: mở rộng `r`, và khi vi phạm điều kiện thì **co** `l`. Kỹ thuật chạy đúng khi điều kiện **đơn điệu**: nếu đoạn thỏa điều kiện thì đoạn *con* của nó cũng thỏa (hoặc ngược lại một cách nhất quán). Ví dụ "có **tối đa 2** ký tự khác nhau" là đơn điệu (bỏ bớt ký tự không làm tăng số loại). Nhưng "mọi ký tự xuất hiện `>= k` lần" **không đơn điệu**: bỏ bớt một ký tự đầu có thể làm tần suất giảm xuống dưới `k`; thêm ký tự có thể sửa được. Ta sẽ giải quyết bằng cách **cố định số ký tự khác nhau** (mục 7).

In [7]:
def longest_at_most_2_distinct(s):
    cnt, l, best = {}, 0, 0
    for r, c in enumerate(s):
        cnt[c] = cnt.get(c, 0) + 1
        while len(cnt) > 2:                  # vi phạm: co từ trái
            d = s[l]
            cnt[d] -= 1
            if cnt[d] == 0:
                del cnt[d]
            l += 1
        best = max(best, r - l + 1)
    return best


print(longest_at_most_2_distinct("eceba"))   # 3 ("ece")

3


### 3.7. Bitmask cho tập ký tự (tham khảo)

Với 26 chữ cái ta có thể biểu diễn một **tập ký tự** bằng số nguyên: bit thứ `i` bằng `1` nếu chữ thứ `i` ở trong tập. `1 << i` tạo bit `i`; `mask |= 1 << i` thêm; `mask & (1 << i)` kiểm tra. Ta dùng nó ở biến thể cuối bài.

In [8]:
mask = 0
for c in "abca":
    mask |= 1 << (ord(c) - ord('a'))
print(bin(mask), bool(mask & (1 << 1)), bool(mask & (1 << 25)))     # 0b111 True False

0b111 True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ví dụ 1: `s = "aaabb"`, `k = 3`.** Đếm cả chuỗi: `a = 3`, `b = 2`. Ký tự `b` chỉ có **2 < 3** lần trong **cả** chuỗi. Vậy *không chuỗi con nào chứa `b`* có thể hợp lệ — vì chuỗi con chỉ chứa **ít hơn hoặc bằng** số `b` của cả chuỗi (tối đa 2 < 3). Vậy mọi chuỗi con hợp lệ **không chứa `b`**: chỉ có thể nằm trong các mảnh khi **cắt tại các `b`**: `"aaa"`. Mảnh này hợp lệ (`a = 3`) ⇒ đáp án `3`.

**Ví dụ 2: `s = "ababbc"`, `k = 2`.** Đếm: `a = 2`, `b = 3`, `c = 1`. Ký tự `c` chỉ có 1 < 2 ⇒ cắt tại `c`: mảnh `"ababb"` (mảnh sau `c` rỗng). Đếm lại trong mảnh: `a = 2`, `b = 3`, cả hai `>= 2` ⇒ hợp lệ, độ dài `5`.

**Quy luật rút ra:** trong một đoạn, nếu có ký tự `ch` xuất hiện `< k` lần thì `ch` là **điểm cắt**: cắt đoạn tại mọi `ch`, giải từng mảnh. Nếu **không** có ký tự nào xuất hiện `< k` lần (mọi ký tự có mặt đều `>= k`) thì **cả đoạn hợp lệ** — đó là đáp án của đoạn.

In [9]:
def show_rare(t, k):
    cnt = Counter(t)
    print(f"đoạn {t!r}: tần suất {dict(cnt)}, ký tự hiếm (< {k}):", [c for c, v in cnt.items() if v < k])


show_rare("aaabb", 3)
show_rare("ababbc", 2)
show_rare("ababb", 2)

đoạn 'aaabb': tần suất {'a': 3, 'b': 2}, ký tự hiếm (< 3): ['b']
đoạn 'ababbc': tần suất {'a': 2, 'b': 3, 'c': 1}, ký tự hiếm (< 2): ['c']
đoạn 'ababb': tần suất {'a': 2, 'b': 3}, ký tự hiếm (< 2): []


## 5. Cách 1 — Brute Force: thử mọi chuỗi con

### Ý tưởng

Với mọi cặp `(l, r)`: lấy `t = s[l:r]`, kiểm tra `is_valid(t, k)` (dùng `Counter`), cập nhật độ dài lớn nhất. Đúng theo định nghĩa đề bài nên rất dễ tin tưởng; ta dùng làm **oracle** cho các cách sau.

In [10]:
def longest_brute(s, k):
    n = len(s)
    best = 0
    for l in range(n):
        for r in range(l + 1, n + 1):
            t = s[l:r]
            if all(v >= k for v in Counter(t).values()):
                best = max(best, r - l)
    return best


for (s, k), expected in examples:
    print(longest_brute(s, k), expected)

3 3
5 5


### Độ phức tạp & nút thắt

- Có `O(n²)` chuỗi con, mỗi chuỗi con tốn `O(n)` để đếm ⇒ **Time `O(n³)`**, Space `O(n)`.
- `n = 10^4` ⇒ cỡ `10^12` thao tác — **không thể**. Kiểm tra thời gian khi `n` tăng:

In [11]:
import time
import random

rng = random.Random(1)
for n in [30, 60, 120]:
    s = "".join(rng.choice("abc") for _ in range(n))
    t0 = time.perf_counter()
    longest_brute(s, 3)
    print(f"n = {n:3}: {time.perf_counter() - t0:.4f}s")

n =  30: 0.0005s
n =  60: 0.0023s
n = 120: 0.0119s


Thời gian tăng khoảng 8 lần mỗi khi `n` gấp đôi (`n³`).

**Nút thắt:** mỗi chuỗi con được đếm lại từ đầu dù chuỗi con kề nhau chỉ khác một ký tự. Ta có thể **cập nhật đếm tăng dần** (Cách 2). Nhưng vẫn `O(n²)`; để đạt `O(n)` cần nhận xét về "ký tự hiếm" (mục 6).

## 6. Ý tưởng thuật toán / bước đột phá

**Nhận xét then chốt:** Xét đoạn `s[l:r]`. Nếu ký tự `ch` xuất hiện **ít hơn `k` lần trong cả đoạn**, thì với mọi chuỗi con của đoạn, số lần `ch` cũng `< k`. Nên chuỗi con chứa `ch` chỉ có thể hợp lệ **nếu số lần `ch` trong nó bằng 0** — tức nó **không chứa `ch`**. Vậy mọi chuỗi con hợp lệ của đoạn nằm trọn trong một mảnh khi cắt đoạn tại **tất cả vị trí của `ch`**.

Thuật toán **chia để trị**:

1. Đếm tần suất trong đoạn.
2. Nếu mọi ký tự có mặt đều `>= k` ⇒ cả đoạn hợp lệ, trả độ dài đoạn.
3. Ngược lại chọn một ký tự hiếm `ch` làm điểm cắt, đệ quy trên từng mảnh, trả về `max`.

**Vì sao nhanh?** Mỗi lần cắt loại hẳn một loại ký tự ra khỏi các mảnh, mà chỉ có **26** loại ⇒ độ sâu đệ quy `≤ 26`. Tại mỗi tầng, các mảnh rời nhau và tổng độ dài `≤ n` ⇒ tổng công việc `O(26 · n)`.

Hướng khác (bước đột phá thứ hai): nếu **cố định số loại ký tự khác nhau trong cửa sổ bằng `t`** thì điều kiện trở nên đơn điệu và dùng được cửa sổ trượt, thử `t` từ `1` đến `26` (Cách 4).

In [12]:
# Xem cây đệ quy "cắt tại ký tự hiếm" (chỉ để quan sát)
def show_split_tree(t, k, depth=0):
    pad = "  " * depth
    cnt = Counter(t)
    rare = [c for c, v in cnt.items() if v < k]
    if not rare:
        print(f"{pad}{t!r}: hợp lệ, độ dài {len(t)}")
        return len(t)
    ch = rare[0]
    print(f"{pad}{t!r}: ký tự hiếm {ch!r} -> cắt thành {t.split(ch)}")
    return max((show_split_tree(piece, k, depth + 1) for piece in t.split(ch) if piece), default=0)


print("kết quả:", show_split_tree("ababbc", 2))
print("kết quả:", show_split_tree("bbaaacbd", 3))

'ababbc': ký tự hiếm 'c' -> cắt thành ['ababb', '']
  'ababb': hợp lệ, độ dài 5
kết quả: 5
'bbaaacbd': ký tự hiếm 'c' -> cắt thành ['bbaaa', 'bd']
  'bbaaa': ký tự hiếm 'b' -> cắt thành ['', '', 'aaa']
    'aaa': hợp lệ, độ dài 3
  'bd': ký tự hiếm 'b' -> cắt thành ['', 'd']
    'd': ký tự hiếm 'd' -> cắt thành ['', '']
kết quả: 3


## 7. Các cách tiếp theo

Tổng cộng **4 cách**: (1) brute `O(n³)`; (2) brute cải tiến `O(n²)` với đếm tăng dần; (3) chia để trị theo ký tự hiếm; (4) cửa sổ trượt với số ký tự khác nhau cố định.

### Cách 2 — Brute cải tiến: đếm tăng dần, biến `bad`

Giữ đầu trái `i` cố định, mở rộng đầu phải `j`, và **cập nhật** mảng đếm khi thêm `s[j]`. Thay vì duyệt mảng đếm để kiểm tra, ta giữ biến `bad` = **số loại ký tự đang có mặt nhưng chưa đủ `k`** (tức `0 < cnt < k`):

- Khi một ký tự lần đầu xuất hiện (`cnt` thành `1`) và `k > 1`: `bad += 1`.
- Khi `cnt` đạt đúng `k` (và `k > 1`): `bad -= 1`.
- Đoạn `[i, j]` hợp lệ khi `bad == 0`.

Mỗi bước `O(1)` ⇒ tổng `O(n²)`.

In [13]:
def longest_incremental(s, k):
    n = len(s)
    best = 0
    for i in range(n):
        cnt = [0] * 26
        bad = 0
        for j in range(i, n):
            idx = ord(s[j]) - 97
            cnt[idx] += 1
            if k > 1:
                if cnt[idx] == 1:
                    bad += 1
                elif cnt[idx] == k:
                    bad -= 1
            if bad == 0:
                best = max(best, j - i + 1)
    return best


for (s, k), expected in examples:
    print(longest_incremental(s, k), expected)

3 3
5 5


**Code nộp LeetCode (Cách 2):**

In [14]:
class SolutionIncremental:
    def longestSubstring(self, s: str, k: int) -> int:
        n = len(s)
        best = 0
        for i in range(n):
            cnt = [0] * 26
            bad = 0
            for j in range(i, n):
                idx = ord(s[j]) - 97
                cnt[idx] += 1
                if k > 1:
                    if cnt[idx] == 1:
                        bad += 1
                    elif cnt[idx] == k:
                        bad -= 1
                if bad == 0:
                    best = max(best, j - i + 1)
        return best


print([SolutionIncremental().longestSubstring(s, k) for (s, k), _ in examples])

[3, 5]


Độ phức tạp: **Time `O(n²)`**, **Space `O(1)`** (mảng 26). Với `n = 10^4` là `~5·10^7` bước: quá chậm trong Python. Chỉ dùng cho `n` nhỏ.

### Cách 3 — Chia để trị theo ký tự hiếm

Giải thích từng đoạn (dùng đoạn nửa mở `[l, r)`):

- `dfs(l, r)`: đếm tần suất của `s[l:r]`.
- Tìm một ký tự `split` có `0 < cnt < k`. Không có ⇒ `return r - l`.
- Ngược lại duyệt đoạn, **bỏ qua** các ký tự `split`, mỗi mảnh liên tiếp không chứa `split` là `[i, j)`: gọi `dfs(i, j)` và lấy `max`.

In [15]:
def longest_divide(s, k):
    def dfs(l, r):
        cnt = Counter(s[l:r])
        split = next((c for c, v in cnt.items() if v < k), None)
        if split is None:
            return r - l
        best = 0
        i = l
        while i < r:
            while i < r and s[i] == split:       # bỏ qua các ký tự cắt
                i += 1
            j = i
            while j < r and s[j] != split:       # mảnh [i, j) không chứa split
                j += 1
            if j > i:
                best = max(best, dfs(i, j))
            i = j
        return best

    return dfs(0, len(s))


for (s, k), expected in examples:
    print(longest_divide(s, k), expected)

3 3
5 5


**Code nộp LeetCode (Cách 3):**

In [16]:
from collections import Counter


class SolutionDivide:
    def longestSubstring(self, s: str, k: int) -> int:
        def dfs(l, r):
            cnt = Counter(s[l:r])
            split = next((c for c, v in cnt.items() if v < k), None)
            if split is None:
                return r - l
            best = 0
            i = l
            while i < r:
                while i < r and s[i] == split:
                    i += 1
                j = i
                while j < r and s[j] != split:
                    j += 1
                if j > i:
                    best = max(best, dfs(i, j))
                i = j
            return best

        return dfs(0, len(s))


print([SolutionDivide().longestSubstring(s, k) for (s, k), _ in examples])

[3, 5]


Độ phức tạp: **Time `O(26 · n)`** (độ sâu ≤ 26, mỗi tầng tổng độ dài ≤ `n`, mỗi đoạn đếm trong `O(độ dài)`), **Space `O(26)` cho đếm + `O(26)` độ sâu đệ quy** (nếu dùng chỉ số thay vì cắt chuỗi mới; chuỗi `s[l:r]` tạm thời tốn `O(độ dài)`).

**Phiên bản ngắn bằng `split`** (rất Pythonic):

```python
def longestSubstring(self, s, k):
    for c in set(s):
        if s.count(c) < k:
            return max(self.longestSubstring(t, k) for t in s.split(c))
    return len(s)
```

In [17]:
class SolutionSplit:
    def longestSubstring(self, s: str, k: int) -> int:
        for c in set(s):
            if s.count(c) < k:
                return max(self.longestSubstring(t, k) for t in s.split(c))
        return len(s)


print([SolutionSplit().longestSubstring(s, k) for (s, k), _ in examples])
print(SolutionSplit().longestSubstring("", 3))        # 0: đoạn rỗng không có ký tự hiếm -> len('') = 0

[3, 5]
0


### Cách 4 — Cửa sổ trượt với số ký tự khác nhau cố định `t`

Với mỗi `t = 1, 2, ..., 26`, tìm đoạn dài nhất có **đúng `t` loại ký tự** và **cả `t` loại đều xuất hiện `>= k` lần**. Đáp án là `max` trên mọi `t`.

Với `t` cố định, ta dùng cửa sổ `[l, r]` và các biến:

- `cnt[c]`: tần suất trong cửa sổ;
- `unique`: số loại ký tự khác nhau trong cửa sổ;
- `enough`: số loại ký tự có `cnt >= k`.

Quy trình: thêm `s[r]` (cập nhật `unique`, `enough`); trong khi `unique > t` thì bỏ `s[l]` (cập nhật ngược) và `l += 1`; nếu `unique == t` và `enough == t` (mọi loại đều đủ) thì cập nhật `best = max(best, r - l + 1)`.

Vì sao đúng: với `t` cố định, điều kiện "`unique <= t`" **đơn điệu** nên cửa sổ trượt đúng; và đoạn tối ưu có `t*` loại ký tự sẽ được tìm đúng ở vòng `t = t*`.

In [18]:
def longest_window(s, k):
    n = len(s)
    best = 0
    max_unique = len(set(s))
    for t in range(1, max_unique + 1):
        cnt = [0] * 26
        unique = enough = 0
        l = 0
        for r in range(n):
            idx = ord(s[r]) - 97
            if cnt[idx] == 0:
                unique += 1
            cnt[idx] += 1
            if cnt[idx] == k:
                enough += 1
            while unique > t:
                jdx = ord(s[l]) - 97
                if cnt[jdx] == k:
                    enough -= 1
                cnt[jdx] -= 1
                if cnt[jdx] == 0:
                    unique -= 1
                l += 1
            if unique == t and enough == t:
                best = max(best, r - l + 1)
    return best


for (s, k), expected in examples:
    print(longest_window(s, k), expected)

3 3
5 5


**Code nộp LeetCode (Cách 4):**

In [19]:
class SolutionWindow:
    def longestSubstring(self, s: str, k: int) -> int:
        n = len(s)
        best = 0
        for t in range(1, len(set(s)) + 1):
            cnt = [0] * 26
            unique = enough = 0
            l = 0
            for r in range(n):
                idx = ord(s[r]) - 97
                if cnt[idx] == 0:
                    unique += 1
                cnt[idx] += 1
                if cnt[idx] == k:
                    enough += 1
                while unique > t:
                    jdx = ord(s[l]) - 97
                    if cnt[jdx] == k:
                        enough -= 1
                    cnt[jdx] -= 1
                    if cnt[jdx] == 0:
                        unique -= 1
                    l += 1
                if unique == t and enough == t:
                    best = max(best, r - l + 1)
        return best


print([SolutionWindow().longestSubstring(s, k) for (s, k), _ in examples])

[3, 5]


Độ phức tạp: **Time `O(26 · n)`** (tối đa 26 vòng `t`, mỗi vòng `O(n)` — mỗi con trỏ đi qua chuỗi một lần), **Space `O(1)`** (mảng 26).

## 8. TRACE từng bước

### Trace Cách 3 (chia để trị) trên `s = "bbaaacbd"`, `k = 3`

| Đoạn | Tần suất | Ký tự hiếm | Hành động |
|---|---|---|---|
| `bbaaacbd` | b=3, a=3, c=1, d=1 | `c` (1 < 3) | cắt tại `c`: mảnh `bbaaa`, `bd` |
| `bbaaa` | b=2, a=3 | `b` (2 < 3) | cắt tại `b`: mảnh `aaa` |
| `aaa` | a=3 | không có | **hợp lệ, 3** |
| `bd` | b=1, d=1 | `b` | cắt tại `b`: mảnh `d` |
| `d` | d=1 | `d` | cắt: không còn mảnh ⇒ `0` |

`max(3, 0) = 3`.

In [20]:
def trace_divide(s, k):
    def dfs(l, r, depth):
        pad = "  " * depth
        seg = s[l:r]
        cnt = Counter(seg)
        split = next((c for c, v in cnt.items() if v < k), None)
        print(f"{pad}dfs[{l},{r}) {seg!r}: đếm={dict(cnt)} -> ", end="")
        if split is None:
            print(f"không có ký tự hiếm => hợp lệ, trả {r - l}")
            return r - l
        print(f"ký tự hiếm {split!r}")
        best = 0
        i = l
        while i < r:
            while i < r and s[i] == split:
                i += 1
            j = i
            while j < r and s[j] != split:
                j += 1
            if j > i:
                best = max(best, dfs(i, j, depth + 1))
            i = j
        print(f"{pad}=> trả {best}")
        return best

    return dfs(0, len(s), 0)


print("kết quả:", trace_divide("bbaaacbd", 3))
print()
print("kết quả:", trace_divide("ababbc", 2))

dfs[0,8) 'bbaaacbd': đếm={'b': 3, 'a': 3, 'c': 1, 'd': 1} -> ký tự hiếm 'c'
  dfs[0,5) 'bbaaa': đếm={'b': 2, 'a': 3} -> ký tự hiếm 'b'
    dfs[2,5) 'aaa': đếm={'a': 3} -> không có ký tự hiếm => hợp lệ, trả 3
  => trả 3
  dfs[6,8) 'bd': đếm={'b': 1, 'd': 1} -> ký tự hiếm 'b'
    dfs[7,8) 'd': đếm={'d': 1} -> ký tự hiếm 'd'
    => trả 0
  => trả 0
=> trả 3
kết quả: 3

dfs[0,6) 'ababbc': đếm={'a': 2, 'b': 3, 'c': 1} -> ký tự hiếm 'c'
  dfs[0,5) 'ababb': đếm={'a': 2, 'b': 3} -> không có ký tự hiếm => hợp lệ, trả 5
=> trả 5
kết quả: 5


### Trace Cách 4 (cửa sổ trượt) trên `s = "ababbc"`, `k = 2`, với `t = 2`

Ký hiệu cửa sổ `[l, r]`; `unique` = số loại; `enough` = số loại có tần suất `>= 2`.

| `r` | `s[r]` | Cửa sổ | Tần suất | `unique` | `enough` | Co trái? | Hợp lệ? (`unique == enough == 2`) | `best` |
|---|---|---|---|---|---|---|---|---|
| 0 | a | `a` | a1 | 1 | 0 | không | không | 0 |
| 1 | b | `ab` | a1 b1 | 2 | 0 | không | không (enough 0) | 0 |
| 2 | a | `aba` | a2 b1 | 2 | 1 | không | không | 0 |
| 3 | b | `abab` | a2 b2 | 2 | 2 | không | **có**, độ dài 4 | 4 |
| 4 | b | `ababb` | a2 b3 | 2 | 2 | không | **có**, độ dài 5 | 5 |
| 5 | c | thêm `c` ⇒ `unique = 3 > 2` nên **co trái**: bỏ `a`, `b`, `a` ⇒ cửa sổ `bbc` | b2 c1 | 2 | 1 | có (bỏ `a`, `b`, `a`) | không (`enough = 1`) | 5 |

Chú ý hàng `r = 5`: sau khi co, `unique` trở về 2 nhưng `c` mới chỉ xuất hiện 1 lần nên `enough = 1`, không bằng 2.

In [21]:
def trace_window(s, k, t):
    cnt = {}
    unique = enough = l = best = 0
    for r, c in enumerate(s):
        if cnt.get(c, 0) == 0:
            unique += 1
        cnt[c] = cnt.get(c, 0) + 1
        if cnt[c] == k:
            enough += 1
        shrunk = []
        while unique > t:
            d = s[l]
            if cnt[d] == k:
                enough -= 1
            cnt[d] -= 1
            if cnt[d] == 0:
                unique -= 1
            shrunk.append(d)
            l += 1
        ok = unique == t and enough == t
        if ok:
            best = max(best, r - l + 1)
        live = {c2: v for c2, v in cnt.items() if v > 0}
        print(f"r={r} s[r]={c!r} cửa sổ={s[l:r + 1]!r:8} đếm={live} unique={unique} enough={enough} "
              f"co={shrunk} hợp lệ={ok} best={best}")
    return best


print("t=1:", trace_window("ababbc", 2, 1))
print()
print("t=2:", trace_window("ababbc", 2, 2))
print()
print("t=3:", trace_window("ababbc", 2, 3))

r=0 s[r]='a' cửa sổ='a'      đếm={'a': 1} unique=1 enough=0 co=[] hợp lệ=False best=0
r=1 s[r]='b' cửa sổ='b'      đếm={'b': 1} unique=1 enough=0 co=['a'] hợp lệ=False best=0
r=2 s[r]='a' cửa sổ='a'      đếm={'a': 1} unique=1 enough=0 co=['b'] hợp lệ=False best=0
r=3 s[r]='b' cửa sổ='b'      đếm={'b': 1} unique=1 enough=0 co=['a'] hợp lệ=False best=0
r=4 s[r]='b' cửa sổ='bb'     đếm={'b': 2} unique=1 enough=1 co=[] hợp lệ=True best=2
r=5 s[r]='c' cửa sổ='c'      đếm={'c': 1} unique=1 enough=0 co=['b', 'b'] hợp lệ=False best=2
t=1: 2

r=0 s[r]='a' cửa sổ='a'      đếm={'a': 1} unique=1 enough=0 co=[] hợp lệ=False best=0
r=1 s[r]='b' cửa sổ='ab'     đếm={'a': 1, 'b': 1} unique=2 enough=0 co=[] hợp lệ=False best=0
r=2 s[r]='a' cửa sổ='aba'    đếm={'a': 2, 'b': 1} unique=2 enough=1 co=[] hợp lệ=False best=0
r=3 s[r]='b' cửa sổ='abab'   đếm={'a': 2, 'b': 2} unique=2 enough=2 co=[] hợp lệ=True best=4
r=4 s[r]='b' cửa sổ='ababb'  đếm={'a': 2, 'b': 3} unique=2 enough=2 co=[] hợp lệ=True best=5


Nhận xét từ trace: ở `t = 2` ta thu được `5` (`"ababb"`); `t = 3` cho `0` vì `c` chỉ xuất hiện 1 lần ⇒ không bao giờ `enough == 3`. Đáp án chung là `max = 5`.

## 9. Code nộp LeetCode

Chọn **Cách 3 (chia để trị)** làm bản nộp chính; ngắn, dễ giải thích:

```python
from collections import Counter

class Solution:
    def longestSubstring(self, s: str, k: int) -> int:
        def dfs(l, r):
            cnt = Counter(s[l:r])
            split = next((c for c, v in cnt.items() if v < k), None)
            if split is None:
                return r - l
            best = 0
            i = l
            while i < r:
                while i < r and s[i] == split:
                    i += 1
                j = i
                while j < r and s[j] != split:
                    j += 1
                if j > i:
                    best = max(best, dfs(i, j))
                i = j
            return best

        return dfs(0, len(s))
```

In [22]:
from collections import Counter


class Solution:
    def longestSubstring(self, s: str, k: int) -> int:
        def dfs(l, r):
            cnt = Counter(s[l:r])
            split = next((c for c, v in cnt.items() if v < k), None)
            if split is None:
                return r - l
            best = 0
            i = l
            while i < r:
                while i < r and s[i] == split:
                    i += 1
                j = i
                while j < r and s[j] != split:
                    j += 1
                if j > i:
                    best = max(best, dfs(i, j))
                i = j
            return best

        return dfs(0, len(s))


for (s, k), expected in examples:
    got = Solution().longestSubstring(s, k)
    print(repr(s), k, got, expected, got == expected)

'aaabb' 3 3 3 True
'ababbc' 2 5 5 True


## 10. Vì sao đúng

**Chia để trị.** Gọi `ans(l, r)` là độ dài chuỗi con hợp lệ dài nhất nằm trong `s[l:r]`.

1. *Nếu mọi ký tự có mặt trong `s[l:r]` đều xuất hiện `>= k` lần*: chính `s[l:r]` hợp lệ và là chuỗi con dài nhất ⇒ `ans = r - l`. Đây là nhánh `split is None`.
2. *Ngược lại tồn tại `ch` với `0 < cnt(ch) < k`:* với mọi chuỗi con `u` của `s[l:r]`, `cnt_u(ch) <= cnt(ch) < k`, nên `u` chỉ hợp lệ nếu `cnt_u(ch) = 0`, tức `u` **không chứa `ch`**. Mọi chuỗi con không chứa `ch` nằm trọn trong một mảnh khi cắt tại các vị trí `ch`. Do đó
   `ans(l, r) = max` trên các mảnh `[i, j)` của `ans(i, j)`. Đó chính là phép đệ quy.

**Kết thúc.** Mỗi mảnh ngắn hơn đoạn cha (đã bỏ ít nhất một ký tự `ch`) nên đệ quy dừng; hơn nữa mảnh **không còn chứa `ch`** nên số loại ký tự giảm ít nhất 1 mỗi tầng ⇒ độ sâu `≤ 26`.

**Cửa sổ trượt (Cách 4).** Cố định `t`. Với mỗi `r`, vòng `while unique > t` đẩy `l` sang phải đúng đến vị trí nhỏ nhất `l(r)` sao cho `s[l(r)..r]` có **không quá `t`** loại ký tự (số loại giảm khi `l` tăng nên `l(r)` không giảm theo `r`, hai con trỏ chỉ đi một chiều). Giả sử `s[a..b]` là đoạn tối ưu có đúng `t` loại, mỗi loại xuất hiện `>= k`. Ở bước `r = b`, vì `s[a..b]` có `<= t` loại nên `l(b) <= a`, tức cửa sổ `s[l(b)..b]` chứa `s[a..b]`. Cửa sổ này có `<= t` loại, mà đã chứa đủ `t` loại của `s[a..b]` nên có đúng `t` loại; mỗi loại xuất hiện nhiều hơn hoặc bằng trong `s[a..b]` nên cũng `>= k` ⇒ `enough == t`, cửa sổ hợp lệ và dài `>=` đoạn tối ưu. Do đó `best` đạt đúng độ dài tối ưu; và mọi lần cập nhật `best` đều ứng với một đoạn hợp lệ thật (có `t` loại, mỗi loại `>= k`), nên không bao giờ vượt quá đáp án.

## 11. Time/Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute (mọi chuỗi con + `Counter`) | `O(n³)` | `O(n)` |
| 2. Brute cải tiến (đếm tăng dần) | `O(n²)` | `O(1)` |
| 3. Chia để trị | **`O(26 · n)`** | `O(26)` đệ quy (+ cắt lát tạm) |
| 4. Cửa sổ trượt `t = 1..26` | **`O(26 · n)`** | `O(1)` |

`26` là kích thước bảng chữ cái, hằng số. Cả Cách 3 và 4 thực chất `O(n)` theo bảng chữ cái cố định. Với `n = 10^4` chỉ cỡ `2.6·10^5` thao tác.

## 12. Bộ test

### 12.1. Ví dụ đề + edge case

In [23]:
methods_small = [longest_brute, longest_incremental, longest_divide, longest_window,
                 SolutionIncremental().longestSubstring, SolutionDivide().longestSubstring,
                 SolutionSplit().longestSubstring, SolutionWindow().longestSubstring,
                 Solution().longestSubstring]

cases = [
    (("aaabb", 3), 3),
    (("ababbc", 2), 5),
    (("a", 1), 1),                       # k = 1: cả chuỗi
    (("a", 2), 0),                       # không đủ
    (("abc", 1), 3),                     # k = 1 luôn là cả chuỗi
    (("abcde", 2), 0),                   # mọi ký tự chỉ 1 lần
    (("aaaaa", 5), 5),
    (("aaaaa", 6), 0),                   # k lớn hơn độ dài
    (("aabbcc", 2), 6),
    (("aaabbb", 3), 6),
    (("bbaaacbd", 3), 3),
    (("ababacb", 3), 0),
    (("weitong", 2), 0),
    (("abacb", 2), 0),
    (("aabcabb", 3), 0),
    (("ababbcbbabab", 3), 0),            # c hiếm; hai mảnh còn lại đều có a hoặc b dưới 3 lần
    (("zzzzzabzzzzz", 5), 5),            # a, b hiếm; hai khối z mỗi khối 5
    (("xxaaaayyy", 3), 7),               # chỉ x hiếm (2 lần): còn khối "aaaayyy", a=4, y=3
]
for (s, k), expected in cases:
    oracle = longest_brute(s, k)
    assert oracle == expected, (s, k, oracle, expected)
    for f in methods_small:
        assert f(s, k) == expected, (f, s, k, expected, f(s, k))
print("OK:", len(cases), "test cố định x", len(methods_small), "cách")

OK: 18 test cố định x 9 cách


Mỗi test được đối chiếu với `longest_brute` (định nghĩa trực tiếp của đề) *và* với số viết tay ở trên.

### 12.2. Ngẫu nhiên có seed: đối chiếu 4 cách với oracle (chuỗi ngắn)

In [24]:
rng = random.Random(395)
count = 0
for _ in range(4000):
    alphabet = rng.choice(["ab", "abc", "abcd", "abcde", "abcdefgh"])
    n = rng.randint(1, 30)
    s = "".join(rng.choice(alphabet) for _ in range(n))
    k = rng.randint(1, 6)
    oracle = longest_brute(s, k)
    for f in methods_small:
        assert f(s, k) == oracle, (f, s, k, oracle)
    count += 1
print("OK:", count, "input ngẫu nhiên, 9 hàm khớp oracle")

OK: 4000 input ngẫu nhiên, 9 hàm khớp oracle


### 12.3. Chuỗi dài hơn: so sánh Cách 2, 3, 4 với nhau

Với `n` vài nghìn, Brute `O(n³)` quá chậm, nên so sánh các cách nhanh với nhau (Cách 2 `O(n²)` vẫn dùng được tới `n ≈ 1500`).

In [25]:
rng = random.Random(3950)
for _ in range(60):
    n = rng.randint(500, 1500)
    alphabet = rng.choice(["ab", "abc", "abcdef", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(rng.choice(alphabet) for _ in range(n))
    k = rng.randint(1, 40)
    r2 = SolutionIncremental().longestSubstring(s, k)
    assert SolutionDivide().longestSubstring(s, k) == r2
    assert SolutionWindow().longestSubstring(s, k) == r2
    assert SolutionSplit().longestSubstring(s, k) == r2
print("OK: 60 chuỗi dài 500..1500")

# n = 10^4, so sánh Cách 3, 4 và phiên bản split
rng = random.Random(10 ** 4)
for _ in range(20):
    alphabet = rng.choice(["ab", "abc", "abcdefghij", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(rng.choice(alphabet) for _ in range(10 ** 4))
    k = rng.choice([1, 2, 5, 50, 300, 1000, 5000])
    a, b, c = (SolutionDivide().longestSubstring(s, k), SolutionWindow().longestSubstring(s, k),
               SolutionSplit().longestSubstring(s, k))
    assert a == b == c, (a, b, c)
print("OK: 20 chuỗi dài 10^4")

OK: 60 chuỗi dài 500..1500


OK: 20 chuỗi dài 10^4


### 12.4. Các chuỗi đặc biệt ở cỡ lớn nhất + hiệu năng

In [26]:
special = [
    ("a" * 10000, 1, 10000),
    ("a" * 10000, 10000, 10000),
    ("a" * 10000, 10001, 0),
    ("a" * 5000 + "b" + "a" * 4999, 2, 5000),       # 'b' hiếm ở giữa: hai khối 'a'
    ("abc" * 3333, 3333, 9999),
    ("abcdefghijklmnopqrstuvwxyz" * 384, 384, 9984),   # 26*384 = 9984
    ("ab" * 5000, 5000, 10000),
]
for s, k, expected in special:
    for f in [SolutionDivide().longestSubstring, SolutionWindow().longestSubstring, SolutionSplit().longestSubstring]:
        assert f(s, k) == expected, (f, len(s), k, expected, f(s, k))
print("OK các chuỗi đặc biệt")

# độ sâu đệ quy lớn nhất: chuỗi mà mỗi tầng chỉ loại được đúng một chữ cái
chars = "abcdefghijklmnopqrstuvwxyz"
deep = "".join(c * (27 - i) for i, c in enumerate(chars))     # 'a' x26, 'b' x25, ... 'z' x1
t0 = time.perf_counter()
vals = [SolutionDivide().longestSubstring(deep, k) for k in range(1, 28)]
print("độ sâu lớn:", vals, round(time.perf_counter() - t0, 4), "giây")

s_big = "".join(random.Random(5).choice("abc") for _ in range(10 ** 4))
for name, f in [("chia để trị", SolutionDivide().longestSubstring), ("cửa sổ", SolutionWindow().longestSubstring),
                ("split", SolutionSplit().longestSubstring)]:
    t0 = time.perf_counter()
    res = f(s_big, 2000)
    print(f"{name:12} {res}  {time.perf_counter() - t0:.4f}s")

OK các chuỗi đặc biệt
độ sâu lớn: [377, 377, 375, 372, 368, 363, 357, 350, 342, 333, 323, 312, 300, 287, 273, 258, 242, 225, 207, 188, 168, 147, 125, 102, 78, 53, 27] 0.0026 giây
chia để trị  10000  0.0003s
cửa sổ       10000  0.0016s
split        10000  0.0000s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: co cửa sổ trực tiếp theo điều kiện "mọi ký tự `>= k`"

In [27]:
def bug_direct_window(s, k):
    cnt = Counter()
    l = best = 0
    for r, c in enumerate(s):
        cnt[c] += 1
        # SAI: co cho đến khi MỌI ký tự có mặt đều >= k (điều kiện không đơn điệu)
        while l <= r and any(0 < v < k for v in cnt.values()):
            cnt[s[l]] -= 1
            l += 1
        best = max(best, r - l + 1)
    return best


print(bug_direct_window("aaabb", 3), "(đúng là", Solution().longestSubstring("aaabb", 3), ")")
print(bug_direct_window("ababbc", 2), "(đúng là", Solution().longestSubstring("ababbc", 2), ")")

0 (đúng là 3 )
0 (đúng là 5 )


**Vì sao sai:** ngay khi cửa sổ mới có `a = 1 < 3` ta co lại hết, nhưng cửa sổ này **sẽ trở nên hợp lệ khi mở rộng thêm** (khi `a` lên `3`). Điều kiện "mọi ký tự `>= k`" không đơn điệu: co lại không đảm bảo "tốt hơn", mở rộng cũng có thể sửa lỗi. **Sửa:** cố định số loại ký tự `t` (Cách 4) hoặc dùng chia để trị (Cách 3).

### Lỗi 2: quên mảnh sau điểm cắt cuối cùng

In [28]:
def bug_miss_last_piece(s, k):
    cnt = Counter(s)
    for c, v in cnt.items():
        if v < k:
            pieces = s.split(c)
            return max((bug_miss_last_piece(p, k) for p in pieces[:-1]), default=0)   # SAI: bỏ mảnh cuối
    return len(s)


print(bug_miss_last_piece("bbaaa", 3), "(đúng là", Solution().longestSubstring("bbaaa", 3), ")")
print(bug_miss_last_piece("ccaaa", 3), "(đúng là", Solution().longestSubstring("ccaaa", 3), ")")

0 (đúng là 3 )
0 (đúng là 3 )


**Vì sao sai:** `"bbaaa".split("b")` cho `['', '', 'aaa']`; mảnh hợp lệ nằm **ở cuối**. Phải xét **mọi** mảnh, kể cả mảnh đầu, giữa và cuối. **Sửa:** `max(... for p in pieces)`.

### Lỗi 3: coi ký tự vắng mặt (tần suất 0) là "hiếm"

In [29]:
import sys


def bug_zero_is_rare(s, k):
    cnt = Counter(s)
    for c in "abcdefghijklmnopqrstuvwxyz":
        if cnt[c] < k:                        # SAI: ký tự vắng mặt (0) cũng < k
            pieces = s.split(c)
            return max(bug_zero_is_rare(p, k) for p in pieces)
    return len(s)


try:
    print(bug_zero_is_rare("aaabb", 3))
except RecursionError:
    print("RecursionError: đệ quy vô hạn vì cắt tại ký tự không có trong chuỗi (split trả lại cả chuỗi)")

RecursionError: đệ quy vô hạn vì cắt tại ký tự không có trong chuỗi (split trả lại cả chuỗi)


**Vì sao sai:** nếu `c` không có trong chuỗi thì `s.split(c) == [s]`, bài toán **không nhỏ đi**, đệ quy lặp mãi. **Sửa:** chỉ xét ký tự có `0 < cnt < k` (đi qua `Counter(s)` chỉ chứa ký tự có mặt, hoặc `set(s)`).

### Lỗi 4: cửa sổ trượt chỉ kiểm tra `unique == t`, quên `enough == t`

In [30]:
def bug_no_enough(s, k):
    best = 0
    for t in range(1, len(set(s)) + 1):
        cnt = Counter()
        l = 0
        for r, c in enumerate(s):
            cnt[c] += 1
            while len([v for v in cnt.values() if v > 0]) > t:
                cnt[s[l]] -= 1
                l += 1
            if len([v for v in cnt.values() if v > 0]) == t:     # SAI: không kiểm tra "mỗi loại >= k"
                best = max(best, r - l + 1)
    return best


print(bug_no_enough("abcde", 2), "(đúng là", Solution().longestSubstring("abcde", 2), ")")
print(bug_no_enough("ababbc", 2), "(đúng là", Solution().longestSubstring("ababbc", 2), ")")

5 (đúng là 0 )
6 (đúng là 5 )


**Vì sao sai:** cửa sổ có đúng `t` loại ký tự chưa chắc mỗi loại đủ `k` lần. Ta còn phải đếm số loại đạt ngưỡng (`enough`) và chỉ nhận cửa sổ khi `enough == t`. **Sửa:** duy trì `enough` và điều kiện `unique == t and enough == t`.

## 14. Mẹo & nhận dạng pattern

- Điều kiện "**mọi ký tự có mặt đều `>= k`**" ⇒ nghĩ ngay tới **ký tự hiếm làm điểm cắt** (chia để trị) hoặc **cửa sổ trượt cố định số loại**.
- Hỏi trước: điều kiện có **đơn điệu** không (bỏ bớt/thêm vào có giữ nguyên tính đúng/sai theo một chiều)? Nếu không, đừng co cửa sổ "thẳng" mà hãy đổi hệ tọa độ (cố định một đại lượng phụ như `t`).
- Bảng chữ cái nhỏ (26) cho phép nhân hằng số 26 vào độ phức tạp: `O(26 · n)` = `O(n)`.
- Chia để trị trên chuỗi: nhớ xét **cả mảnh đầu, giữa, cuối** và mảnh rỗng.
- Dùng `s.count(c)` và `s.split(c)` cho code ngắn gọn (như `SolutionSplit`) — rất hợp để nộp nhanh.
- Họ hàng: *Longest Substring Without Repeating Characters*, *Longest Substring with At Most K Distinct Characters*, *Longest Repeating Character Replacement*, *Subarrays with K Different Integers*, *Maximum Size Subarray Sum Equals k*.

## 15. Biến thể

1. **Tối đa `k` ký tự khác nhau** (LeetCode 340): cửa sổ trượt đơn điệu, dùng một lần duyệt.
2. **Trả về chính chuỗi con** thay vì độ dài: giữ `(độ dài, vị trí đầu)` trong chia để trị.
3. **Ký tự bất kỳ (Unicode)**, không giới hạn 26: Cách chia để trị vẫn đúng (độ sâu `≤` số loại ký tự), cửa sổ cố định `t` thì cần `t` chạy tới số loại ký tự.
4. **Đếm số chuỗi con hợp lệ** thay vì độ dài lớn nhất: khó hơn, thường dùng brute cho `n` nhỏ.
5. **Ngưỡng khác nhau cho từng ký tự** (`k[c]`): chia để trị sửa điều kiện ký tự hiếm thành `cnt[c] < k[c]`.

Minh họa biến thể 1 và 2:

In [31]:
def longest_at_most_k_distinct(s, k):
    cnt, l, best = {}, 0, 0
    for r, c in enumerate(s):
        cnt[c] = cnt.get(c, 0) + 1
        while len(cnt) > k:
            d = s[l]
            cnt[d] -= 1
            if cnt[d] == 0:
                del cnt[d]
            l += 1
        best = max(best, r - l + 1)
    return best


print(longest_at_most_k_distinct("eceba", 2), longest_at_most_k_distinct("aa", 1))      # 3 2


def longest_substring_text(s, k):
    """Trả về chính chuỗi con hợp lệ dài nhất (nếu hòa, lấy mảnh xuất hiện đầu tiên)."""
    def dfs(l, r):
        cnt = Counter(s[l:r])
        split = next((c for c, v in cnt.items() if v < k), None)
        if split is None:
            return (r - l, l)
        best = (0, 0)
        i = l
        while i < r:
            while i < r and s[i] == split:
                i += 1
            j = i
            while j < r and s[j] != split:
                j += 1
            if j > i:
                cand = dfs(i, j)
                if cand[0] > best[0]:
                    best = cand
            i = j
        return best

    length, start = dfs(0, len(s))
    return s[start:start + length]


print(repr(longest_substring_text("ababbc", 2)), repr(longest_substring_text("aaabb", 3)), repr(longest_substring_text("abc", 2)))
rng = random.Random(15)
for _ in range(500):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 25)))
    k = rng.randint(1, 4)
    t = longest_substring_text(s, k)
    assert len(t) == longest_brute(s, k) and (t == "" or is_valid(t, k)) and t in s
print("OK biến thể trả về chuỗi con")

3 2
'ababb' 'aaa' ''


OK biến thể trả về chuỗi con


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute `O(n³)` | Mọi chuỗi con + `Counter` | `O(n³)` | `O(n)` | Làm **oracle** cho `n` rất nhỏ |
| 2. Brute cải tiến | Đếm tăng dần + biến `bad` | `O(n²)` | `O(1)` | `n` vài trăm/nghìn, dễ hiểu |
| 3. Chia để trị | Cắt tại ký tự hiếm, đệ quy | **`O(26 n)`** | `O(26)` | **Cách nộp chuẩn**, hợp mọi bảng chữ cái |
| 4. Cửa sổ trượt `t=1..26` | Cố định số loại để có đơn điệu | **`O(26 n)`** | `O(1)` | Khi cần một lượt không đệ quy; bảng chữ cái nhỏ |

## 17. Bài tập vận dụng (dễ → khó)

### Bài 1 (đoán output)

Đoán `guess_1()` trả về gì (một tuple gồm một list và một list) trước khi chạy.

In [32]:
def guess_1():
    t = "aabxaab"
    cnt = Counter("aabbc")
    return t.split("x"), sorted(c for c, v in cnt.items() if v < 2)


answer_1 = None   # TODO: điền một tuple (list, list)

### Bài 2 (tính tay)

Với `s = "bbaaacbd"`, `k = 3`: tính **bằng tay** (chia để trị, ghi từng đoạn) đáp án. Điền `answer_2`.

In [33]:
answer_2 = None   # TODO: điền một số nguyên

### Bài 3 (điền TODO)

Hoàn thiện hàm chia để trị bằng `split`.

In [34]:
def longest_split_todo(s, k):
    # TODO 1: duyệt từng ký tự c có mặt trong s (gợi ý: set(s))
    # TODO 2: nếu s.count(c) < k thì c là điểm cắt: trả về max đáp án của các mảnh s.split(c)
    # TODO 3: nếu không có ký tự hiếm thì cả chuỗi hợp lệ
    ...

### Bài 4 (tự viết lại)

Viết `is_valid_fast(t, k)` kiểm tra mọi ký tự có mặt trong `t` xuất hiện `>= k` lần **chỉ dùng `str.count`** (không dùng `Counter`). Ví dụ `is_valid_fast("ababb", 2) is True`, `is_valid_fast("aaabb", 3) is False`, `is_valid_fast("", 3) is True`.

In [35]:
def is_valid_fast(t, k):
    # TODO: tự viết
    pass

### Bài 5 (biến thể / mini challenge)

Viết `longest_at_most_k_distinct_todo(s, k)`: độ dài chuỗi con dài nhất có **tối đa `k` ký tự khác nhau** (cửa sổ trượt). Ví dụ `("eceba", 2) → 3`, `("aa", 1) → 2`, `("abc", 0) → 0`. Gợi ý: nhớ trường hợp `k = 0`.

In [36]:
def longest_at_most_k_distinct_todo(s, k):
    # TODO: tự viết
    pass

## 18. Gợi ý (3 mức)

**Bài 1**
- Mức 1: `"aabxaab".split("x")` cắt tại `x`.
- Mức 2: `Counter("aabbc")` là `a=2, b=2, c=1`; ký tự có tần suất `< 2` chỉ có `c`.
- Mức 3: kết quả `(['aab', 'aab'], ['c'])`.

**Bài 2**
- Mức 1: đếm: `b=3, a=3, c=1, d=1`; ký tự hiếm `c`.
- Mức 2: cắt tại `c`: `bbaaa` và `bd`. Trong `bbaaa` có `b=2 < 3` nên cắt tại `b`.
- Mức 3: ra mảnh `aaa` hợp lệ (3); `bd` cho `0`. Đáp án `3`.

**Bài 3**
- Mức 1: `for c in set(s):`.
- Mức 2: `if s.count(c) < k: return max(longest_split_todo(p, k) for p in s.split(c))`.
- Mức 3: sau vòng lặp `return len(s)`. (Chuỗi rỗng ⇒ `len('') = 0`.)

**Bài 4**
- Mức 1: duyệt `for c in set(t)`.
- Mức 2: mỗi `c` kiểm tra `t.count(c) >= k`.
- Mức 3: `return all(t.count(c) >= k for c in set(t))`.

**Bài 5**
- Mức 1: giống cửa sổ `longest_at_most_2_distinct` nhưng thay `2` bằng `k`.
- Mức 2: `while len(cnt) > k` co trái; xóa khóa khi đếm về `0`.
- Mức 3: nếu `k == 0` thì `while` sẽ co đến khi cửa sổ rỗng; `best` giữ `0` (hoặc trả sớm `0`).

## 19. Đáp án tham khảo

In [37]:
# Bài 1
answer_1 = (['aab', 'aab'], ['c'])
assert guess_1() == answer_1

# Bài 2
answer_2 = 3
assert answer_2 == Solution().longestSubstring("bbaaacbd", 3) == longest_brute("bbaaacbd", 3)


# Bài 3
def longest_split_todo(s, k):
    for c in set(s):
        if s.count(c) < k:
            return max(longest_split_todo(p, k) for p in s.split(c))
    return len(s)


rng = random.Random(19)
for _ in range(500):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 20)))
    k = rng.randint(1, 4)
    assert longest_split_todo(s, k) == longest_brute(s, k)


# Bài 4
def is_valid_fast(t, k):
    return all(t.count(c) >= k for c in set(t))


assert is_valid_fast("ababb", 2) is True
assert is_valid_fast("aaabb", 3) is False
assert is_valid_fast("", 3) is True
assert all(is_valid_fast(t, 2) == is_valid(t, 2) for t in ["", "a", "aa", "aab", "aabb", "abab", "ababbc"])


# Bài 5
def longest_at_most_k_distinct_todo(s, k):
    if k == 0:
        return 0
    cnt, l, best = {}, 0, 0
    for r, c in enumerate(s):
        cnt[c] = cnt.get(c, 0) + 1
        while len(cnt) > k:
            d = s[l]
            cnt[d] -= 1
            if cnt[d] == 0:
                del cnt[d]
            l += 1
        best = max(best, r - l + 1)
    return best


assert longest_at_most_k_distinct_todo("eceba", 2) == 3
assert longest_at_most_k_distinct_todo("aa", 1) == 2
assert longest_at_most_k_distinct_todo("abc", 0) == 0
for _ in range(300):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 15)))
    k = rng.randint(1, 4)
    brute = max(len(s[l:r]) for l in range(len(s)) for r in range(l + 1, len(s) + 1) if len(set(s[l:r])) <= k)
    assert longest_at_most_k_distinct_todo(s, k) == brute
print("Tất cả đáp án đã chạy đúng")

Tất cả đáp án đã chạy đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (chia để trị bằng `split`)

```python
def longestSubstring(self, s, k):
    for c in set(s):
        if s.count(c) < k:             # c là ký tự hiếm => điểm cắt
            return max(self.longestSubstring(t, k) for t in s.split(c))
    return len(s)                      # mọi ký tự có mặt đều >= k => cả chuỗi hợp lệ
```

### Template (cửa sổ trượt cố định số loại `t`)

```python
best = 0
for t in range(1, len(set(s)) + 1):
    cnt = [0] * 26; unique = enough = l = 0
    for r in range(len(s)):
        add s[r]  -> cập nhật unique, enough
        while unique > t: bỏ s[l] -> cập nhật ngược; l += 1
        if unique == t and enough == t: best = max(best, r - l + 1)
return best
```

### Checklist

- [ ] Ký tự cắt phải là ký tự **có mặt** với `0 < cnt < k` (không phải ký tự vắng mặt).
- [ ] Chia để trị xét **mọi mảnh** (đầu, giữa, cuối), kể cả mảnh rỗng (cho `0`).
- [ ] Nhánh cơ sở: không có ký tự hiếm ⇒ trả độ dài đoạn.
- [ ] Cửa sổ trượt: cố định `t`; kiểm `unique == t` **và** `enough == t`.
- [ ] Không co cửa sổ trực tiếp theo điều kiện "mọi ký tự `>= k`" (không đơn điệu).
- [ ] Test: `("aaabb", 3) → 3`, `("ababbc", 2) → 5`, `k = 1`, `k > len(s)`, chuỗi toàn một ký tự, `n = 10^4`.